# Appendix 7. Feature Engineering 기초

## 1. Goal

이 notebook은 EDA finding을 재현 가능한 candidate feature와 검토 evidence로 바꾸는 흐름을 익히는 자료입니다. Long-format 관측값을 patient-level로 집계하고, derived feature와 missing indicator를 만든 뒤 correlation, mutual information, 단변량 Logistic Regression, coefficient, tree importance와 permutation importance를 비교합니다.

여기서 기초는 feature API 목록이 아니라 feature representation과 selection의 원리를 뜻합니다. 어떤 점수 하나로 feature를 자동 선택하지 않으며, prediction 시점의 가용성, leakage, 결측, redundancy, validation 안정성과 feature contract를 함께 검토합니다. 각 방법이 무엇을 측정하고 어떤 조건에서 오해를 만드는지 설명하는 것이 목표입니다.

## 2. Setup

pandas, NumPy, Matplotlib과 scikit-learn을 사용합니다. 실제 PhysioNet data나 canonical feature set은 변경하지 않고, 48시간 observation window를 흉내 낸 합성 long-format data를 고정 seed로 생성합니다. 모든 association과 importance는 합성 관계를 설명할 뿐 임상적 결론이 아닙니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

BLUE = "#2F5D8C"
ORANGE = "#D97732"
INK = "#30343B"
GRID = "#D9DEE5"
RANDOM_SEED = 42
OBSERVATION_WINDOW_MINUTE = 48 * 60


## 3. Steps

### 3-1. Feature 생성 경계

#### 3-1-1. Prediction 시점과 observation window

feature는 prediction 시점에 관측 가능한 정보로만 계산해야 합니다. 이 예제는 입실 후 48시간까지를 observation window로 사용하고, 그 이후의 `FutureOutcome` 관측값은 leakage 사례로만 남깁니다. target과 강한 관계를 보이더라도 가용성 검사를 통과하지 못하면 model 비교 전에 제외합니다.

In [ ]:
def make_synthetic_longitudinal_data(
    patient_count: int, seed: int
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return deterministic patient metadata and longitudinal observations."""
    rng = np.random.default_rng(seed)
    patient_rows: list[dict[str, object]] = []
    observation_rows: list[dict[str, object]] = []
    minutes = [0, 720, 1440, 2160, 2880]

    for index in range(patient_count):
        patient_id = f"P{index:04d}"
        age = float(np.clip(rng.normal(63, 15), 18, 92))
        icu_type = str(rng.choice(["medical", "surgical", "cardiac"]))
        base_heart_rate = float(rng.normal(86, 13))
        base_lactate = float(rng.lognormal(0.65, 0.45))
        urine_rate = float(rng.lognormal(5.3, 0.35))
        risk_signal = (
            0.035 * (age - 60)
            + 0.05 * (base_heart_rate - 85)
            + 0.9 * (base_lactate - 2)
            - 0.002 * (urine_rate - 200)
            + 0.25 * (icu_type == "medical")
            + rng.normal(0, 0.8)
        )
        target_probability = 1 / (1 + np.exp(-risk_signal))
        target = int(rng.binomial(1, target_probability))
        patient_rows.append(
            {
                "patient_id": patient_id,
                "age": age,
                "icu_type": icu_type,
                "noise_feature": float(rng.normal()),
                "constant_feature": 1.0,
                "target": target,
            }
        )

        lactate_is_missing = rng.random() < 0.14
        for position, minute in enumerate(minutes):
            observation_rows.append(
                {
                    "patient_id": patient_id,
                    "minute": minute,
                    "parameter": "HR",
                    "value": base_heart_rate + 1.8 * position + rng.normal(0, 3),
                }
            )
            observation_rows.append(
                {
                    "patient_id": patient_id,
                    "minute": minute,
                    "parameter": "Urine",
                    "value": max(0.0, urine_rate + rng.normal(0, 25)),
                }
            )
            if not lactate_is_missing:
                observation_rows.append(
                    {
                        "patient_id": patient_id,
                        "minute": minute,
                        "parameter": "Lactate",
                        "value": max(0.1, base_lactate + rng.normal(0, 0.2)),
                    }
                )
        observation_rows.append(
            {
                "patient_id": patient_id,
                "minute": 60 * 60,
                "parameter": "FutureOutcome",
                "value": target + rng.normal(0, 0.03),
            }
        )

    return pd.DataFrame(patient_rows), pd.DataFrame(observation_rows)


patient_table, observations = make_synthetic_longitudinal_data(220, RANDOM_SEED)
print({"patients": patient_table.shape, "observations": observations.shape})
observations.head()


### 3-2. Candidate feature 생성

#### 3-2-1. Long-format 관측값 집계

시간순으로 정렬한 뒤 patient와 parameter별 `min`, `max`, `mean`, `first`, `last`, `count`, `sum`을 계산합니다. 같은 통계를 모든 parameter에 적용한 것은 API 실습을 단순하게 하기 위한 것입니다. 실제 aggregation plan은 parameter별 의미와 단위를 기준으로 versioned config에 정의합니다.

In [ ]:
within_window = (
    observations.loc[observations["minute"].le(OBSERVATION_WINDOW_MINUTE)]
    .sort_values(["patient_id", "parameter", "minute"])
)
aggregated = (
    within_window.groupby(["patient_id", "parameter"])["value"]
    .agg(["min", "max", "mean", "first", "last", "count", "sum"])
    .unstack("parameter")
)
aggregated.columns = [
    f"{parameter.lower()}__{statistic}"
    for statistic, parameter in aggregated.columns
]
aggregated = aggregated.reset_index()
print({"max_observation_minute": within_window["minute"].max()})
aggregated.head()


#### 3-2-2. Derived feature와 missing indicator

기본 집계에서 range, 변화량과 missing indicator를 파생합니다. `count`는 관측량을, `__missing`은 관측이 전혀 없음을 나타내므로 서로 다른 정보를 담습니다. ratio나 slope를 만들 때는 0인 분모, 시간 중복, 단위와 값의 방향을 먼저 정의해야 합니다.

In [ ]:
candidate_features = patient_table.merge(aggregated, on="patient_id", how="left")
candidate_features = candidate_features.assign(
    heart_rate__range=lambda frame: frame["hr__max"] - frame["hr__min"],
    heart_rate__delta=lambda frame: frame["hr__last"] - frame["hr__first"],
    lactate__missing=lambda frame: frame["lactate__count"].isna().astype("int8"),
    heart_rate__mean_copy=lambda frame: frame["hr__mean"] * 1.02,
)
future_feature = (
    observations.loc[observations["parameter"].eq("FutureOutcome")]
    .set_index("patient_id")["value"]
    .rename("future_outcome__last")
)
candidate_features = candidate_features.join(future_feature, on="patient_id")
candidate_features[
    ["patient_id", "hr__mean", "lactate__max", "urine__sum", "lactate__missing"]
].head()


### 3-3. Model 이전의 기본 screening

#### 3-3-1. Availability, 결측률과 variance

Target association보다 먼저 prediction 시점의 availability를 확인합니다. 이후 결측률, unique count와 variance로 사용할 수 없는 후보를 찾습니다. 이런 unsupervised screening은 target을 보지 않지만 정보가 적다는 이유만으로 중요한 rare signal을 제거할 수도 있습니다. Threshold는 예제 안에서 즉흥적으로 정하지 않고 실제 project의 versioned feature policy가 소유해야 합니다.

In [ ]:
candidate_columns = [
    "age",
    "hr__min",
    "hr__max",
    "hr__mean",
    "hr__last",
    "hr__count",
    "lactate__max",
    "lactate__mean",
    "lactate__count",
    "urine__sum",
    "heart_rate__range",
    "heart_rate__delta",
    "lactate__missing",
    "heart_rate__mean_copy",
    "noise_feature",
    "constant_feature",
    "future_outcome__last",
]
feature_profile = pd.DataFrame(
    {
        "missing_rate": candidate_features[candidate_columns].isna().mean(),
        "unique_count": candidate_features[candidate_columns].nunique(dropna=True),
        "variance": candidate_features[candidate_columns].var(numeric_only=True),
        "available_at_prediction": [
            column != "future_outcome__last" for column in candidate_columns
        ],
    }
)
eligible_columns = feature_profile.query(
    "available_at_prediction and unique_count > 1 and missing_rate < 0.5"
).index.tolist()
feature_profile.sort_values(["available_at_prediction", "unique_count"]).head()


#### 3-3-2. Correlation으로 redundancy 후보 찾기

Feature-feature correlation은 거의 같은 정보를 담은 후보를 찾는 screening 도구입니다. 높은 pairwise correlation은 redundancy 후보를 알려 주지만, 한 feature가 다른 여러 feature의 조합으로 설명되는 multicollinearity를 모두 찾지는 못합니다. 높은 correlation만으로 자동 제거하지 않고 source 정의, 결측과 안정성을 비교해 대표 feature를 고릅니다. 여기서는 의도적으로 만든 heart-rate copy를 제거합니다.

In [ ]:
eligible_correlation = candidate_features[eligible_columns].corr(method="spearman").abs()
upper_triangle = eligible_correlation.where(
    np.triu(np.ones(eligible_correlation.shape), k=1).astype(bool)
)
high_correlation_pairs = (
    upper_triangle.stack()
    .rename("absolute_spearman")
    .loc[lambda values: values.gt(0.95)]
    .sort_values(ascending=False)
)
redundant_columns = ["heart_rate__mean_copy"]
model_candidate_columns = [
    column for column in eligible_columns if column not in redundant_columns
]
high_correlation_pairs.head(8)


#### 3-3-3. Filter, wrapper, embedded와 inspection

Feature selection 방법은 목적과 model 의존성에 따라 구분할 수 있습니다. Filter method는 model 학습 전 통계량으로 빠르게 후보를 줄입니다. Wrapper method는 feature subset마다 model을 반복 평가해 계산량과 overfitting 위험이 큽니다. Embedded method는 L1 regularization이나 tree split처럼 model fitting 과정에서 선택 효과를 냅니다. Model inspection은 이미 fitted된 model이 feature에 의존하는 방식을 설명하며 selection rule과 동일하지 않습니다.

Supervised selector도 data에서 학습하는 transformer입니다. Cross-validation 밖에서 전체 data로 먼저 selection하면 validation fold의 target 정보가 training fold로 새어 들어가므로 selector를 Pipeline 안에 넣어야 합니다.

In [ ]:
selection_method_guide = pd.DataFrame(
    [
        {
            "family": "filter",
            "examples": "variance, correlation, mutual information",
            "strength": "fast and model-independent",
            "main_limit": "often ignores interactions",
        },
        {
            "family": "wrapper",
            "examples": "RFE, sequential selection",
            "strength": "evaluates a concrete estimator and metric",
            "main_limit": "expensive and selection can overfit CV",
        },
        {
            "family": "embedded",
            "examples": "L1 coefficient, tree split",
            "strength": "selection occurs during fitting",
            "main_limit": "depends on model assumptions",
        },
        {
            "family": "inspection",
            "examples": "coefficient, impurity, permutation",
            "strength": "explains fitted-model dependence",
            "main_limit": "importance is not causal relevance",
        },
    ]
)
selection_method_guide


### 3-4. Target association을 여러 방법으로 비교

#### 3-4-1. Train/validation 분리

target을 사용하는 correlation, mutual information, regression과 importance 계산은 train data에서 수행합니다. validation은 선택한 workflow가 새로운 sample에서도 동작하는지 확인하는 데 남겨 둡니다. sealed test는 feature 판단에 사용하지 않습니다.

In [ ]:
train_features, valid_features = train_test_split(
    candidate_features,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=candidate_features["target"],
)
X_train = train_features[model_candidate_columns]
y_train = train_features["target"]
X_valid = valid_features[model_candidate_columns]
y_valid = valid_features["target"]
print(
    {
        "train_shape": X_train.shape,
        "valid_shape": X_valid.shape,
        "train_target_rate": y_train.mean(),
        "valid_target_rate": y_valid.mean(),
    }
)


#### 3-4-2. Pearson, Spearman과 mutual information

Pearson은 선형 관계, Spearman은 단조 관계를 요약합니다. Mutual information은 더 일반적인 statistical dependency를 탐색할 수 있지만 방향을 제공하지 않고 sample이 적으면 추정이 불안정할 수 있습니다. 연속형 값을 이산형으로 잘못 지정하는 등 estimator 설정에 따라서도 값이 달라집니다. 세 값은 scale이 다르므로 절대값을 직접 같은 단위처럼 비교하지 않고 ranking과 방향을 함께 봅니다.

In [ ]:
pearson = X_train.corrwith(y_train, method="pearson")
spearman = X_train.corrwith(y_train, method="spearman")
association_imputer = SimpleImputer(strategy="median")
X_train_imputed = association_imputer.fit_transform(X_train)
mutual_information = pd.Series(
    mutual_info_classif(
        X_train_imputed, y_train, random_state=RANDOM_SEED
    ),
    index=model_candidate_columns,
    name="mutual_information",
)
association_table = pd.DataFrame(
    {
        "pearson": pearson,
        "spearman": spearman,
        "mutual_information": mutual_information,
    }
).sort_values("mutual_information", ascending=False)
association_table.head(10)


#### 3-4-3. 단변량 Logistic Regression

binary target에는 feature 하나씩 Logistic Regression Pipeline을 fit해 cross-validated ROC AUC를 비교할 수 있습니다. 이는 input-output의 predictive association을 보는 방법이지 correlation coefficient가 아닙니다. 연속 target이라면 `LinearRegression`, `r_regression` 또는 `f_regression`처럼 문제 유형에 맞는 방법을 사용합니다.

In [ ]:
cross_validation = StratifiedKFold(
    n_splits=4, shuffle=True, random_state=RANDOM_SEED
)


def univariate_logistic_auc(feature_name: str) -> tuple[float, float]:
    """Return mean and standard deviation of one-feature CV ROC AUC."""
    estimator = make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    )
    scores = cross_val_score(
        estimator,
        X_train[[feature_name]],
        y_train,
        cv=cross_validation,
        scoring="roc_auc",
    )
    return float(scores.mean()), float(scores.std())


univariate_scores = pd.DataFrame.from_dict(
    {
        feature: univariate_logistic_auc(feature)
        for feature in model_candidate_columns
    },
    orient="index",
    columns=["univariate_cv_roc_auc", "univariate_cv_std"],
).sort_values("univariate_cv_roc_auc", ascending=False)
univariate_scores.head(10)


### 3-5. Model-based importance

#### 3-5-1. Standardized Logistic Regression coefficient

모든 numeric feature를 median imputation과 scaling한 뒤 Logistic Regression을 fit합니다. Scaling 후 coefficient의 절대값은 같은 model 안에서 비교하기 쉬워지지만 importance의 절대 척도는 아닙니다. 값은 regularization 강도, class balance, interaction 누락과 correlated feature의 영향을 받습니다. Coefficient 부호는 다른 feature를 고정한 조건부 방향이므로 단변량 correlation과 다를 수 있습니다.

In [ ]:
logistic_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
)
logistic_pipeline.fit(X_train, y_train)
logistic_probabilities = logistic_pipeline.predict_proba(X_valid)[:, 1]
logistic_coefficients = pd.Series(
    logistic_pipeline.named_steps["logisticregression"].coef_[0],
    index=model_candidate_columns,
    name="logistic_coefficient",
)
print({"valid_roc_auc": roc_auc_score(y_valid, logistic_probabilities)})
logistic_coefficients.reindex(
    logistic_coefficients.abs().sort_values(ascending=False).index
).head(10)


#### 3-5-2. Tree impurity importance

Random Forest의 `feature_importances_`는 tree split에서 감소시킨 impurity를 누적한 model-specific 값입니다. Nonlinear 관계와 interaction을 반영할 수 있지만 split 후보가 많은 continuous·high-cardinality feature에 유리할 수 있고 correlated feature 사이에 importance가 나뉩니다. Training 과정에서 계산되므로 generalization 기여를 직접 보여 주지 않으며, model family가 달라지면 importance의 의미도 달라집니다.

In [ ]:
forest_pipeline = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=False),
    RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=4,
        random_state=RANDOM_SEED,
        n_jobs=1,
    ),
)
forest_pipeline.fit(X_train, y_train)
forest_probabilities = forest_pipeline.predict_proba(X_valid)[:, 1]
tree_importance = pd.Series(
    forest_pipeline.named_steps["randomforestclassifier"].feature_importances_,
    index=model_candidate_columns,
    name="tree_importance",
)
print({"valid_roc_auc": roc_auc_score(y_valid, forest_probabilities)})
tree_importance.sort_values(ascending=False).head(10)


#### 3-5-3. Validation permutation importance

Permutation importance는 fitted model의 한 feature를 섞어 feature-target 관계를 깨뜨렸을 때 validation score가 얼마나 감소하는지 측정합니다. Unseen data에서 계산할 수 있고 model 종류와 무관하지만 model 자체의 validation 성능이 충분하지 않으면 해석할 기반이 약합니다. Correlated feature가 대체 정보를 제공하면 중요도가 작게 보일 수 있고, 섞은 값 조합이 현실에 존재하지 않는 sample을 만들 수도 있습니다. Scoring metric과 반복 표준편차를 함께 확인합니다.

In [ ]:
permutation_result = permutation_importance(
    logistic_pipeline,
    X_valid,
    y_valid,
    scoring="roc_auc",
    n_repeats=12,
    random_state=RANDOM_SEED,
    n_jobs=1,
)
permutation_table = pd.DataFrame(
    {
        "permutation_mean": permutation_result.importances_mean,
        "permutation_std": permutation_result.importances_std,
    },
    index=model_candidate_columns,
).sort_values("permutation_mean", ascending=False)

plot_table = permutation_table.head(8).sort_values("permutation_mean")
axis = plot_table["permutation_mean"].plot.barh(
    xerr=plot_table["permutation_std"],
    figsize=(8, 4.2),
    color=BLUE,
    edgecolor=INK,
    capsize=3,
)
axis.axvline(0, color=INK, linewidth=1)
axis.set(
    title="Validation permutation importance",
    xlabel="Mean ROC AUC decrease",
    ylabel="Feature",
)
axis.grid(axis="x", color=GRID, linewidth=0.8)
plt.show()
permutation_table.head(10)


### 3-6. Evidence를 feature decision으로 연결

#### 3-6-1. 여러 방법의 rank 비교

각 방법은 다른 질문에 답합니다. correlation은 pairwise 관계, 단변량 model은 feature 하나의 predictive association, coefficient는 linear model의 conditional weight, tree importance는 fitted forest의 split 기여, permutation은 validation score 의존도를 보여줍니다. rank가 일치하는지와 왜 다른지를 검토합니다.

In [ ]:
importance_comparison = (
    association_table.join(univariate_scores)
    .join(logistic_coefficients)
    .join(tree_importance)
    .join(permutation_table)
)
ranking_inputs = pd.DataFrame(
    {
        "abs_pearson": importance_comparison["pearson"].abs(),
        "spearman": importance_comparison["spearman"].abs(),
        "mutual_information": importance_comparison["mutual_information"],
        "univariate_auc": importance_comparison["univariate_cv_roc_auc"],
        "abs_logistic_coefficient": importance_comparison["logistic_coefficient"].abs(),
        "tree_importance": importance_comparison["tree_importance"],
        "permutation_importance": importance_comparison["permutation_mean"],
    }
)
method_ranks = ranking_inputs.rank(ascending=False, method="average")
importance_comparison["mean_method_rank"] = method_ranks.mean(axis=1)
importance_comparison.sort_values("mean_method_rank").head(10)


#### 3-6-2. Selection stability와 uncertainty

Feature ranking은 sample과 split이 바뀌면 달라질 수 있습니다. 단변량 CV score의 fold 간 표준편차와 permutation 반복 표준편차는 서로 다른 uncertainty를 보여 주므로 숫자 크기를 직접 비교하지 않습니다. 대신 top feature의 방향과 순위가 여러 fold, seed와 validation cohort에서 유지되는지 확인합니다.

많은 후보와 selection 전략을 반복 비교하면 validation data에도 점차 맞춰질 수 있습니다. Selection policy와 primary metric을 먼저 고정하고, 최종 판단은 제한된 validation 과정에서 수행하며 sealed test는 마지막 한 번의 평가에 남겨 둡니다.

In [ ]:
stability_summary = importance_comparison[
    ["univariate_cv_roc_auc", "univariate_cv_std", "permutation_mean", "permutation_std"]
].copy()
stability_summary["permutation_signal_to_noise"] = (
    stability_summary["permutation_mean"]
    / stability_summary["permutation_std"].replace(0, np.nan)
)
stability_summary.reindex(
    importance_comparison.sort_values("mean_method_rank").index
).head(10)


#### 3-6-3. Feature contract와 선택 기록

최종 결정은 점수표만이 아니라 feature provenance와 contract를 포함해야 합니다. 아래는 합성 예제를 위한 작은 contract입니다. 실제 feature subset 변경은 train/CV와 validation evidence만 사용하고 config, DVC revision, model metadata를 함께 갱신합니다.

In [ ]:
screened_features = (
    importance_comparison.sort_values("mean_method_rank").head(6).index.tolist()
)
feature_contract = pd.DataFrame(
    {
        "name": screened_features,
        "dtype": "float",
        "nullable": [candidate_features[name].isna().any() for name in screened_features],
        "available_at_prediction": True,
        "selection_scope": "synthetic train/CV and validation only",
    }
)
feature_contract


## 4. Checks

Observation window, leakage 제거, 기본 screening, importance shape와 feature contract invariant를 확인합니다.

In [ ]:
assert within_window["minute"].max() == OBSERVATION_WINDOW_MINUTE
assert candidate_features["patient_id"].is_unique
assert "future_outcome__last" not in eligible_columns
assert "constant_feature" not in eligible_columns
assert "heart_rate__mean_copy" not in model_candidate_columns
assert "target" not in model_candidate_columns
assert set(association_table) == {"pearson", "spearman", "mutual_information"}
assert len(permutation_table) == len(model_candidate_columns)
assert feature_contract["name"].is_unique
assert feature_contract["available_at_prediction"].all()
assert len(selection_method_guide) == 4
assert set(stability_summary) == {
    "univariate_cv_roc_auc",
    "univariate_cv_std",
    "permutation_mean",
    "permutation_std",
    "permutation_signal_to_noise",
}
print("All appendix checks passed.")


## 5. Next Steps

- Feature 후보는 prediction 시점과 observation window를 통과한 값으로만 만듭니다.
- correlation, mutual information, 단변량 model과 importance를 서로 다른 evidence로 비교합니다.
- importance가 높아도 leakage, unstable measurement, 과도한 결측이나 운영 불가 feature는 제외합니다.
- correlated feature에서는 importance가 나뉠 수 있으므로 정의와 안정성을 기준으로 대표를 고릅니다.
- feature 생성·선택 규칙, column 순서와 dtype을 versioned contract로 고정합니다.
- sealed test 결과를 보고 feature를 다시 추가하거나 제거하지 않습니다.

## 6. References

이 notebook의 설명과 예제는 다음 공식 문서를 기준으로 작성했습니다.

- [pandas GroupBy](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [pandas.DataFrame.corr](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)
- [Feature selection](https://scikit-learn.org/stable/modules/feature_selection.html)
- [mutual_info_classif](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html)
- [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Permutation feature importance](https://scikit-learn.org/stable/modules/permutation_importance.html)
- [Feature importances with a forest](https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html)
- [Common pitfalls and data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)